In [67]:
#Генерация ключа

In [68]:
F = GF(2)
r = 1 #порядок
m = 3 #число переменных
C = codes.BinaryReedMullerCode(r, m)

In [69]:
n = C.length()
k = C.dimension()
d = C.minimum_distance()
t = 2^(m - r - 1)

In [70]:
n,k,d

(8, 4, 4)

In [71]:
def random_permutation_matrix(n, F):
    perm = list(range(n))
    shuffle(perm)
    P = matrix(F, n, n, lambda i, j: 1 if j == perm[i] else 0)
    return P, perm

In [72]:
P, perm = random_permutation_matrix(n, F)
P

[0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 1 0]
[0 0 0 1 0 0 0 0]
[0 0 1 0 0 0 0 0]
[0 0 0 0 0 0 0 1]
[0 1 0 0 0 0 0 0]
[0 0 0 0 1 0 0 0]
[1 0 0 0 0 0 0 0]

In [73]:
G = C.generator_matrix()
print("Порождающая матрица G:")
print(G)

Порождающая матрица G:
[1 1 1 1 1 1 1 1]
[0 1 0 1 0 1 0 1]
[0 0 1 1 0 0 1 1]
[0 0 0 0 1 1 1 1]


In [74]:
def random_invertible_matrix(k, F):
    while True:
        A = random_matrix(F, k, k)
        if A.is_invertible():
            return A

In [75]:
A = random_invertible_matrix(k, F)
A

[0 1 0 0]
[1 1 0 0]
[1 1 0 1]
[0 0 1 0]

In [76]:
E = A * G * P

In [77]:
E

[1 1 1 0 0 0 1 0]
[0 0 0 1 1 1 0 1]
[1 1 0 1 0 1 0 0]
[1 0 1 1 1 0 0 0]

In [78]:
pubkey = (E, t)
pubkey

(
[1 1 1 0 0 0 1 0]   
[0 0 0 1 1 1 0 1]   
[1 1 0 1 0 1 0 0]   
[1 0 1 1 1 0 0 0], 2
)

In [79]:
privkey = (A,G,P)
privkey

(
                              [0 0 0 0 0 1 0 0]
                              [0 0 0 0 0 0 1 0]
                              [0 0 0 1 0 0 0 0]
                              [0 0 1 0 0 0 0 0]
[0 1 0 0]  [1 1 1 1 1 1 1 1]  [0 0 0 0 0 0 0 1]
[1 1 0 0]  [0 1 0 1 0 1 0 1]  [0 1 0 0 0 0 0 0]
[1 1 0 1]  [0 0 1 1 0 0 1 1]  [0 0 0 0 1 0 0 0]
[0 0 1 0], [0 0 0 0 1 1 1 1], [1 0 0 0 0 0 0 0]
)

In [100]:
#Шифрование
msg = "hi"
# Переводим строку в битовую строку
bits = ''.join(format(ord(c), '08b') for c in msg)
print("Биты:", bits)  # '0110100001101001'

# Дополняем нулями до кратности k (если нужно)
if len(bits) % k != 0:
    bits = bits.ljust(len(bits) + (k - len(bits) % k), '0')  # padding нулями справа

# Разбиваем на блоки и создаём вектора
blocks = [bits[i:i+k] for i in range(0, len(bits), k)]
vectors = [vector(F, [int(b) for b in block]) for block in blocks]

print(f"\nРазбито на {len(vectors)} блок(-а/ов):")
for i, v in enumerate(vectors, 1):
    print(f"m_{i} =", v)

Биты: 0110100001101001

Разбито на 4 блок(-а/ов):
m_1 = (0, 1, 1, 0)
m_2 = (1, 0, 0, 0)
m_3 = (0, 1, 1, 0)
m_4 = (1, 0, 0, 1)


In [101]:
a_arr = []
for m in vectors:
    a_arr.append(m * E)

In [148]:
from random import sample

def random_error_vector(n, t, F=GF(2)):
    """
    #ДО t - 1
    Генерирует случайный вектор ошибок длины n над полем F,
    содержащий ровно t единиц (0 ≤ t ≤ n).
    
    Parameters:
        n (int): длина вектора
        t (int): вес (число единиц), 0 ≤ t ≤ n
        F (Field): конечное поле (по умолчанию GF(2))
    
    Returns:
        vector: вектор ошибок e ∈ F^n, wt(e) = t
    """

    e = vector(F, n)
    if t == 0:
        return e
    # Гарантированно выбрать t различных позиций — без циклов и коллизий
    positions = sample(range(n), t-1)
    for i in positions:
        e[i] = F(1)
    return e

In [149]:
e_arr = []
for e in range(len(vectors)):
    e_arr.append(random_error_vector(n, t, F))

In [150]:
b_arr = []
for i in range(len(e_arr)):
    b_arr.append(a_arr[i] + e_arr[i])

In [151]:
def invert_permutation(perm):
    inv = [0] * len(perm)
    for i, j in enumerate(perm):
        inv[j] = i
    return inv

In [152]:
assert len(a_arr) == len(e_arr) == len(b_arr) == len(vectors), \
    "Количество блоков не совпадает!"

In [153]:
b_arr

[(1, 1, 0, 0, 1, 1, 0, 1),
 (1, 1, 1, 0, 0, 0, 0, 0),
 (1, 1, 0, 0, 1, 0, 0, 0),
 (1, 1, 0, 1, 1, 0, 1, 0)]

In [154]:
#Расшифрование

In [155]:
b_hat_arr = []
P_inv = P.inverse()
for b in b_arr:
    b_hat_arr.append(b*P_inv)

In [156]:
b_hat_arr

[(1, 0, 0, 0, 1, 1, 1, 1),
 (0, 0, 0, 1, 0, 1, 0, 1),
 (0, 0, 0, 0, 0, 1, 1, 1),
 (0, 1, 1, 0, 0, 1, 1, 1)]

In [157]:
decoder = C.decoder("NearestNeighbor")  # или "Syndrome", но NearestNeighbor надёжнее для малых k

m_A_arr = []
for i, b_hat in enumerate(b_hat_arr, 1):
    try:
        m_A = decoder.decode_to_message(b_hat)  # ← без распаковки!
        m_A_arr.append(m_A)
        print(f"Блок {i}: успешно → m_A = {list(m_A)}")
    except Exception as e:
        print(f"Блок {i}: ОШИБКА — {e}")
        raise

Блок 1: успешно → m_A = [0, 0, 0, 1]
Блок 2: успешно → m_A = [0, 1, 0, 0]
Блок 3: успешно → m_A = [0, 0, 0, 1]
Блок 4: успешно → m_A = [0, 1, 1, 0]


In [158]:
m_A_arr

[(0, 0, 0, 1), (0, 1, 0, 0), (0, 0, 0, 1), (0, 1, 1, 0)]

In [159]:
m_dec = []
A_inv = A.inverse()
for a in m_A_arr:
    m_dec.append(a*A_inv)

In [160]:
m_dec

[(0, 1, 1, 0), (1, 0, 0, 0), (0, 1, 1, 0), (1, 0, 0, 1)]

In [161]:
vectors

[(0, 1, 1, 0), (1, 0, 0, 0), (0, 1, 1, 0), (1, 0, 0, 1)]

In [162]:
def vectors_to_string(vec_list, k=4):
    # 1. Биты
    bits = ''.join(''.join(str(int(b)) for b in v) for v in vec_list)
    # 2. Байты
    chars = []
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        if len(byte) < 8:
            byte = byte.ljust(8, '0')  # дополняем нулями, если нужно
        chars.append(chr(int(byte, 2)))
    return ''.join(chars)

# Применение:
recovered = vectors_to_string(m_dec)
print("Расшифровано:", recovered)  # hi

Расшифровано: hi
